# Week 9: Atomic Simulation Environment (ASE) 기초 및 Band Gap 계산
이 노트북에서는 ASE(Atomic Simulation Environment)를 활용하여 원자 구조(Bulk Silicon)를 생성하고, DFT 계산기(GPAW 예시)를 사용하여 Band Gap과 Band Structure를 계산하는 방법을 실습합니다.

## 1. 필수 라이브러리 설치 및 임포트
계산을 위해 `ase` 패키지가 필요합니다. 밴드 구조 계산을 위해 `gpaw`도 사용할 수 있습니다.

In [ ]:
!pip install ase
!pip install gpaw # GPAW는 C 컴파일러 의존성이 있으므로 환경에 맞게 별도 설치가 필요할 수 있습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.build import bulk
from ase.visualize import view

## 2. Bulk Silicon 구조 생성
ASE의 `ase.build` 모듈에 있는 `bulk` 함수를 사용하면 다양한 결정 구조를 쉽게 생성할 수 있습니다. 다이아몬드 구조를 가진 Silicon(Si)을 생성해보겠습니다.

In [ ]:
# Si 결정 생성 (다이아몬드 구조, 격자 상수 a=5.43 Angstrom)
si = bulk('Si', 'diamond', a=5.43)

# 생성된 구조 정보 확인
print("화학식:", si.get_chemical_formula())
print("Cell Vectors (Angstrom):\n", si.get_cell())
print("원자 위치 (Angstrom):\n", si.get_positions())
print("주기적 경계 조건 (PBC):", si.pbc)

생성된 구조는 `ase.io.write`를 사용하여 PNG 이미지 파일로 저장하고, Matplotlib을 통해 인라인으로 확인할 수 있습니다.

In [ ]:
from ase.io import write
import matplotlib.image as mpimg

# 구조를 PNG 이미지 파일로 저장 (원하는 회전 각도 적용)
write('si_structure.png', si, rotation='10z,-80x')

# Jupyter Notebook 내에서 PNG 파일 보여주기
img = mpimg.imread('si_structure.png')
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img)
ax.axis('off')
plt.show()

## 3. 구조 최적화 및 Ground State 에너지 계산 (GPAW 사용)
밴드 구조(Band Structure)와 밴드 갭(Band Gap)을 계산하기 전에, 구조의 전자 상태를 수렴시켜야 합니다. ASE는 다양한 계산기(Calculator)를 지원하며, 여기서는 오픈소스 DFT 코드인 **GPAW**를 사용하는 방법을 살펴봅니다.

*(주의: 아래 코드는 실제 GPAW가 설치된 환경에서만 동작합니다.)*

In [ ]:

from gpaw import GPAW, PW

# 평면파(Plane Wave) 기반 계산기 설정
calc = GPAW(mode=PW(300),                    # Cutoff energy: 300 eV
            kpts={'size': (4, 4, 4), 'gamma': True}, # k-point grid (4x4x4)
            xc='PBE',                        # Exchange-Correlation functional
            txt='si_gs.txt')                 # 계산 로그 파일 저장

# Si 구조에 계산기 연결
si.calc = calc

# Ground state 전자 구조 수렴 및 총 에너지 계산
energy = si.get_potential_energy()
print(f"Total Energy: {energy:.3f} eV")

# 계산 결과를 파일로 저장 (다음 단계에서 불러오기 위함)
calc.write('si_gs.gpw')


## 4. Band Gap 및 Band Structure 계산
Ground state의 전하 밀도(electron density)가 수렴된 후, 이를 고정(fixed density)하고 밴드 구조를 그리기 위한 특정 k-point 경로(예: G-X-W-K-G-L)를 따라 고유값(eigenvalues)을 계산합니다.

In [ ]:

# Si의 Brillouin zone을 따르는 k-point 경로 생성
# G(Gamma) -> X -> W -> K -> G -> L 경로 설정
kpts = {'path': 'GXWKGL', 'npoints': 100}

# 저장된 gpw 파일을 불러와서 Non-Self-Consistent(NSCF) 계산 수행
calc_band = GPAW('si_gs.gpw').fixed_density(
    kpts=kpts,
    symmetry='off',
    txt='si_bs.txt'
)
si.calc = calc_band

# Band path 상의 에너지 고유값 계산
si.get_potential_energy()

# 밴드 구조 객체 가져오기
bs = calc_band.band_structure()

# Band Gap 추출
from ase.dft.bandgap import bandgap
gap, p1, p2 = bandgap(calc_band)
print(f"\n계산된 Band Gap: {gap:.3f} eV")

# 밴드 구조 그래프 그리기
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)
bs.plot(ax=ax, filename='Si_bandstructure.png', show=True)


위의 코드를 실행하면 Si의 **간접 천이(indirect) Band Gap** 값(대략 0.6 ~ 0.7 eV, PBE functional 특성상 실험값 1.1 eV보다 과소평가됨)을 얻을 수 있으며, k-point 경로에 따른 **Band Structure 그래프**가 출력됩니다.